# RoadScan YOLO Training on RDD2022 (fixed)

**What changed vs. the original notebook**
- Download is **verified** (size check, clear errors) instead of failing silently; uses the figshare API rather than a guessed URL.
- Can use RDD2022 **attached as a Kaggle dataset** (no download needed) – auto-detected.
- Extraction handles **nested zips** and any folder layout (`Country/train/images` + `annotations/xmls`).
- **Real train/val split** (the original used the same images for both).
- Hard stops with helpful messages if conversion produces 0 images.
- Label fixes: box clipping, degenerate-box removal, no bare `except`.

**Before running on Kaggle:** either attach an RDD2022 dataset (Add Data → search "RDD2022"), *or* turn **Internet → On** in notebook settings.

In [ ]:
!pip install -q -U ultralytics

In [ ]:
import os, json, shutil, random, zipfile
from pathlib import Path
from collections import Counter
import torch

print(f'CUDA available: {torch.cuda.is_available()}')

# ----------------------------- CONFIG -----------------------------
IS_KAGGLE = Path('/kaggle/working').exists()
BASE_DIR  = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()

WORK_DIR   = BASE_DIR / 'roadscan'
DL_DIR     = WORK_DIR / 'download'   # downloaded zips (deleted after extraction)
RAW_DIR    = WORK_DIR / 'raw'        # extracted dataset (only used when downloading)
YOLO_DIR   = WORK_DIR / 'yolo'       # converted YOLO dataset
MODELS_DIR = WORK_DIR / 'models'

# Option A: path of an RDD2022 dataset you attached on Kaggle, e.g. '/kaggle/input/rdd2022'
#           Leave None to auto-detect under /kaggle/input, else it will be downloaded.
KAGGLE_INPUT_DIR = None
FIGSHARE_ARTICLE_ID = 21431547

KEEP_CLASSES = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}
CLASS_NAMES  = list(KEEP_CLASSES.keys())

VAL_SPLIT = 0.15
SEED      = 42

MODEL     = 'yolo11s.pt'
EPOCHS    = 30     # 1 = smoke test (validated on Kaggle), 30 = full run
IMGSZ     = 640
BATCH     = 16
PATIENCE  = 10
MAX_HOURS = 10.5   # stop training after 10.5h (Kaggle sessions cap at ~12h). Overrides EPOCHS. Set None to use EPOCHS only.
# -------------------------------------------------------------------

for d in [WORK_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Get RDD2022

In [ ]:
def has_voc_annotations(root):
    for _, _, files in os.walk(root):
        if any(f.lower().endswith('.xml') for f in files):
            return True
    return False


def download_figshare(article_id, dest_dir):
    import requests
    from tqdm import tqdm
    dest_dir.mkdir(parents=True, exist_ok=True)
    try:
        meta = requests.get(f'https://api.figshare.com/v2/articles/{article_id}', timeout=60)
        meta.raise_for_status()
    except requests.RequestException as e:
        raise RuntimeError(
            'Could not reach figshare. On Kaggle, enable Internet in the notebook settings '
            f'(or attach RDD2022 via Add Data instead). Original error: {e}')
    files = meta.json()['files']
    print('Files in article:', [(f['name'], round(f['size'] / 1e9, 2)) for f in files], '(name, GB)')
    for f in files:
        out = dest_dir / f['name']
        if out.exists() and out.stat().st_size == f['size']:
            print(f"  already downloaded: {f['name']}")
            continue
        with requests.get(f['download_url'], stream=True, timeout=60, allow_redirects=True) as r:
            r.raise_for_status()
            with open(out, 'wb') as fh, tqdm(total=f['size'], unit='B', unit_scale=True, desc=f['name']) as bar:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
                    bar.update(len(chunk))
        got = out.stat().st_size
        if got != f['size']:
            raise RuntimeError(f"Incomplete download for {f['name']}: {got} of {f['size']} bytes")


def extract_all_zips(src_dir, dest_dir, delete_zips=True):
    # Extract every zip under src_dir (and any zips nested inside) into dest_dir.
    dest_dir.mkdir(parents=True, exist_ok=True)
    pending = sorted(Path(src_dir).rglob('*.zip'))
    while pending:
        for zf in pending:
            print(f'  extracting {zf.name}')
            with zipfile.ZipFile(zf) as z:
                z.extractall(dest_dir)
            if delete_zips:
                zf.unlink()
        pending = sorted(dest_dir.rglob('*.zip'))   # nested zips


DOWNLOADED = False
if KAGGLE_INPUT_DIR:
    RAW_DIR = Path(KAGGLE_INPUT_DIR)
elif Path('/kaggle/input').exists() and has_voc_annotations('/kaggle/input'):
    RAW_DIR = Path('/kaggle/input')
else:
    print('No attached dataset found -> downloading from figshare')
    download_figshare(FIGSHARE_ARTICLE_ID, DL_DIR)
    print('Extracting...')
    extract_all_zips(DL_DIR, RAW_DIR)
    shutil.rmtree(DL_DIR, ignore_errors=True)
    DOWNLOADED = True

print('RAW_DIR =', RAW_DIR)
assert RAW_DIR.exists() and has_voc_annotations(RAW_DIR), (
    f'No Pascal-VOC .xml annotations found under {RAW_DIR}. '
    'Check the dataset path / download.')

## 2. Convert to YOLO (with train/val split)

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

IMG_EXTS = {'.jpg', '.jpeg', '.png'}


def voc_to_yolo(xml_path, w, h):
    try:
        root = ET.parse(xml_path).getroot()
    except (ET.ParseError, OSError):
        return []
    lines = []
    for obj in root.findall('object'):
        cls = (obj.findtext('name') or '').strip()
        if cls not in KEEP_CLASSES:
            continue
        bb = obj.find('bndbox')
        if bb is None:
            continue
        try:
            x1 = float(bb.findtext('xmin', '0')); y1 = float(bb.findtext('ymin', '0'))
            x2 = float(bb.findtext('xmax', '0')); y2 = float(bb.findtext('ymax', '0'))
        except ValueError:
            continue
        # clip to image and drop degenerate boxes
        x1, x2 = max(0.0, min(x1, w)), max(0.0, min(x2, w))
        y1, y2 = max(0.0, min(y1, h)), max(0.0, min(y2, h))
        if x2 - x1 < 1 or y2 - y1 < 1:
            continue
        xc, yc = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
        bw, bh = (x2 - x1) / w, (y2 - y1) / h
        lines.append(f'{KEEP_CLASSES[cls]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines

In [ ]:
def collect_samples(raw_root):
    # Find every folder containing .xml files that has a sibling images folder.
    # Works for Country/train/annotations/xmls + Country/train/images and for VOC layouts.
    samples, per_split = [], Counter()
    for dirpath, _, files in os.walk(raw_root):
        if not any(f.lower().endswith('.xml') for f in files):
            continue
        xml_dir = Path(dirpath)
        candidates = [xml_dir.parent.parent / 'images', xml_dir.parent / 'images',
                      xml_dir.parent.parent / 'JPEGImages', xml_dir.parent / 'JPEGImages',
                      xml_dir]
        img_dir = next((c for c in candidates if c.is_dir() and
                        any(p.suffix.lower() in IMG_EXTS for p in c.iterdir())), None)
        if img_dir is None:
            continue
        try:
            tag = '_'.join(img_dir.parent.relative_to(raw_root).parts) or 'root'
        except ValueError:
            tag = 'root'
        images = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
        for xml_file in xml_dir.glob('*.xml'):
            img = images.get(xml_file.stem)
            if img is not None:
                samples.append((tag, img, xml_file))
                per_split[tag] += 1
    return samples, per_split


def place(src, dst):
    # hard-link when on the same filesystem (no extra disk), otherwise copy
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

In [ ]:
samples, per_split = collect_samples(RAW_DIR)
print('Annotated image/xml pairs found per folder:')
for k, v in sorted(per_split.items()):
    print(f'  {k}: {v}')
assert samples, 'No image/annotation pairs found - check folder layout of the dataset.'

# fresh output dir (avoids stale files from earlier failed runs)
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for s in ('train', 'val'):
    (YOLO_DIR / 'images' / s).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / s).mkdir(parents=True, exist_ok=True)

random.Random(SEED).shuffle(samples)
n_val = int(len(samples) * VAL_SPLIT)
splits = {'val': samples[:n_val], 'train': samples[n_val:]}

stats = {s: Counter() for s in splits}
n_done = {s: 0 for s in splits}
for split, items in splits.items():
    for tag, img_path, xml_path in tqdm(items, desc=split):
        try:
            with Image.open(img_path) as im:
                w, h = im.size
        except (OSError, ValueError):
            continue
        lines = voc_to_yolo(xml_path, w, h)
        stem = f'{tag}_{img_path.stem}'
        place(img_path, YOLO_DIR / 'images' / split / (stem + img_path.suffix.lower()))
        (YOLO_DIR / 'labels' / split / (stem + '.txt')).write_text('\n'.join(lines))
        n_done[split] += 1
        for ln in lines:
            stats[split][CLASS_NAMES[int(ln.split()[0])]] += 1

print('\nImages:', n_done)
for s in stats:
    print(f'{s} instances per class:', dict(stats[s]))
assert n_done['train'] > 0 and n_done['val'] > 0, 'Conversion produced no images.'

if DOWNLOADED:                       # free disk space (hard links keep the converted copies)
    shutil.rmtree(RAW_DIR, ignore_errors=True)

In [ ]:
import yaml
data_cfg = {
    'path':  str(YOLO_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(CLASS_NAMES),
    'names': {i: n for i, n in enumerate(CLASS_NAMES)},
}
DATA_YAML = YOLO_DIR / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
print(DATA_YAML.read_text())

## 3. Train YOLO11

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
train_args = dict(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(MODELS_DIR), name='roadscan', exist_ok=True,
    patience=PATIENCE, workers=2, seed=SEED, plots=True,
)
if MAX_HOURS:
    train_args['time'] = MAX_HOURS       # hours; overrides epochs

results = model.train(**train_args)
SAVE_DIR = Path(model.trainer.save_dir)
BEST_PT = SAVE_DIR / 'weights' / 'best.pt'
print('Best weights:', BEST_PT, BEST_PT.exists())

## 4. Evaluate (best checkpoint, validation split)

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ)

print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')
per_class = {}
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    per_class[CLASS_NAMES[int(c)]] = {'precision': float(p), 'recall': float(r),
                                      'mAP50': float(ap50), 'mAP50-95': float(ap)}
    print(f'  {CLASS_NAMES[int(c)]}: P={p:.3f} R={r:.3f} mAP50={ap50:.3f} mAP50-95={ap:.3f}')

## 5. Export & package

In [ ]:
onnx_path = best_model.export(format='onnx', simplify=True, imgsz=IMGSZ)
shutil.copy(BEST_PT, MODELS_DIR / 'best.pt')
shutil.copy(onnx_path, MODELS_DIR / Path(onnx_path).name)
print('Saved best.pt and', Path(onnx_path).name)

metrics_json = {
    'mAP50': float(metrics.box.map50), 'mAP50-95': float(metrics.box.map),
    'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
    'per_class': per_class,
}
(MODELS_DIR / 'metrics.json').write_text(json.dumps(metrics_json, indent=2))
print(json.dumps(metrics_json, indent=2))

zip_path = shutil.make_archive(str(WORK_DIR / 'results'), 'zip', root_dir=WORK_DIR, base_dir='models')
print('Done! Download:', zip_path)